# 특허 도면 유사도 검색 - 결과 데모 (OpenCV)

특허 도면 이미지를 넣으면, 비슷하게 생긴 다른 특허 도면을 찾아 순위로 보여줍니다.
- 전처리 → 특징 추출(ORB) → 특징점 매칭 → 유사도 순위
- 아래 셀을 위에서부터 실행하세요. 각 셀의 출력을 그대로 캡처해 발표 자료로 쓸 수 있습니다.

## 1. 준비

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # OpenMP 런타임 충돌 회피

import cv2
import matplotlib.pyplot as plt

from search_drawings import search

# 경로 설정
IMAGE_DIR = os.path.join("..", "images")          # 특허 도면 폴더 (파일명 = 출원번호)
QUERY = os.path.join("..", "..", "test.jpg")       # 넣을 쿼리 도면

print("쿼리 이미지:", os.path.abspath(QUERY))
print("도면 폴더  :", os.path.abspath(IMAGE_DIR))
print("폴더 내 도면 수:", len([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]))

: 

## 2. 넣을 도면(쿼리) 미리보기

In [ ]:
q = cv2.imread(QUERY)
q = cv2.cvtColor(q, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 6))
plt.imshow(q)
plt.title("Query drawing (test.jpg)")
plt.axis("off")
plt.show()

## 3. 유사 도면 검색 실행 → 순위 출력

In [ ]:
query_name, top = search(QUERY, IMAGE_DIR, topk=5)

print(f"쿼리 도면: {query_name}")
print(f"유사 도면 top-{len(top)} (OpenCV ORB 특징점 매칭 기준)\n")
for i, r in enumerate(top, 1):
    title = r["title"] or "(제목 정보 없음)"
    print(f"{i:>2}위  [유사도 {r['score']:.4f}]  {r['patent_id']}  (매칭 {r['good_matches']}개)")
    print(f"      → {title}")

## 4. 결과를 표로 정리

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"순위": i, "출원번호": r["patent_id"], "유사도": r["score"],
     "매칭 수": r["good_matches"], "제목": r["title"]}
    for i, r in enumerate(top, 1)
])
df

## 5. 쿼리 + 상위 결과 도면 나란히 보기

In [ ]:
def load_rgb(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

topn = min(3, len(top))
fig, axes = plt.subplots(1, topn + 1, figsize=(5 * (topn + 1), 6))

# 쿼리
axes[0].imshow(load_rgb(QUERY))
axes[0].set_title("Query (test.jpg)", fontsize=13)
axes[0].axis("off")

# 상위 결과
for idx in range(topn):
    r = top[idx]
    path = os.path.join(IMAGE_DIR, r["file"])
    axes[idx + 1].imshow(load_rgb(path))
    axes[idx + 1].set_title(f"{idx+1}위  sim={r['score']:.3f}\n{r['patent_id']}", fontsize=12)
    axes[idx + 1].axis("off")

plt.tight_layout()
plt.show()